# Demo 04 : Data and content pipelines

We query the service database for a repeat fault, then turn sourced procedure details into a checked draft bulletin. Run the cells in order and watch. There is nothing to write.

Run the next cell first. It loads the course key and builds `llm`, the course model. Nothing below it depends on where the notebook runs.

In [1]:
# Course setup : run this cell first.
# On the classroom laptop the key comes from .env in the repository root.
# In Colab, add it under the key icon in the left sidebar as COURSE_API_KEY.
import os
import sys
import subprocess
import warnings
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
COURSE_FILES = "https://raw.githubusercontent.com/FeikoWielsma/Multi-Agents-LangGraph/main/"
ROOT = None

if IN_COLAB:
    # Pinned to match uv.lock ; generated by build/stamp_setup.py
    PINS = ["langchain[mcp]==1.4.5", "langchain-core==1.6.10", "langchain-openai==1.7.1", "langgraph==1.2.15", "langgraph-checkpoint-sqlite==3.1.2", "deepagents==0.7.24", "langsmith==0.14.8", "openai==3.29.0", "python-dotenv==1.2.4", "langchain-huggingface==1.2.3", "sentence-transformers==6.1.0", "langchain-text-splitters==1.1.3", "rank-bm25==0.2.2", "agentevals==0.0.9", "httpx==0.28.1", "pytest==9.1.1", "pandas==3.0.6"]
    if not os.getenv("COURSE_SKIP_INSTALL"):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)
    try:
        from google.colab import userdata
        os.environ.setdefault("COURSE_API_KEY", userdata.get("COURSE_API_KEY"))
    except Exception:
        pass
else:
    from dotenv import load_dotenv
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "pyproject.toml").exists():
            ROOT = folder
            load_dotenv(folder / ".env")
            break

PROVIDER = os.getenv("COURSE_PROVIDER", "course")      # "course" or "ollama"
if PROVIDER == "ollama":
    MODEL = os.getenv("COURSE_MODEL", "qwen3.5:4b")
    BASE_URL = "http://localhost:11434/v1"
    API_KEY = "ollama"
    THINKING_OFF = {"reasoning_effort": "none"}
else:
    MODEL = os.getenv("COURSE_MODEL", "openai/gpt-6-luna")  # xiaomi/mimo-v2.6-pro is the step up
    BASE_URL = os.getenv("COURSE_BASE_URL", "https://openrouter.ai/api/v1")
    API_KEY = os.getenv("COURSE_API_KEY")
    THINKING_OFF = {"reasoning": {"effort": "none"}}
    if not API_KEY:
        raise RuntimeError("COURSE_API_KEY is missing : copy .env.example to .env and paste the key")

from langchain_core._api import LangChainBetaWarning
warnings.filterwarnings("ignore", category=LangChainBetaWarning)

from langchain_openai import ChatOpenAI
from langchain_core.rate_limiters import InMemoryRateLimiter

# New OpenRouter accounts may send 20 requests a minute per model.
# One shared limiter keeps parallel agents under it ; raise COURSE_RPS when the cap lifts
RATE_LIMITER = InMemoryRateLimiter(requests_per_second=float(os.getenv("COURSE_RPS", "0.3")),
                                   check_every_n_seconds=0.1, max_bucket_size=1)


def make_llm(**kw):
    """The course chat model. Keyword arguments pass straight to ChatOpenAI."""
    kw.setdefault("model", MODEL)
    kw.setdefault("temperature", 0)
    kw.setdefault("extra_body", THINKING_OFF)
    kw.setdefault("timeout", 90)        # seconds per request ; a stuck call raises instead of hanging
    kw.setdefault("rate_limiter", RATE_LIMITER)
    return ChatOpenAI(base_url=BASE_URL, api_key=API_KEY, **kw)


def make_embeddings(**kw):
    """Local sentence embeddings, so retrieval costs no API calls."""
    from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


def data(name):
    """Path to a course data file ; downloaded when there is no local copy."""
    if ROOT and (ROOT / "data" / name).exists():
        return str(ROOT / "data" / name)
    target = Path(name)
    if not target.exists():
        import urllib.request
        target.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(COURSE_FILES + "data/" + name, target)
    return str(target)


def data_folder(name):
    """Path to a course data folder ; downloaded file by file, using its index.txt, when there is no local copy."""
    if ROOT and (ROOT / "data" / name).is_dir():
        return str(ROOT / "data" / name)
    for file in Path(data(f"{name}/index.txt")).read_text(encoding="utf-8").split():
        data(f"{name}/{file}")
    return name


llm = make_llm()
print(f"model {MODEL} via {PROVIDER}")

model openai/gpt-6-luna via course


## a · data-analysis

We compute the ranking in SQL and ask the model to explain the returned rows. The database result stays visible for verification.

In [2]:
import sqlite3

db_path = data("demo-world/service.db")
with sqlite3.connect(f"file:{Path(db_path).as_posix()}?mode=ro", uri=True) as db:
    rows = db.execute("SELECT error_code, COUNT(*) AS work_orders "
                      "FROM work_orders GROUP BY error_code ORDER BY COUNT(*) DESC LIMIT 5").fetchall()
print("query rows:", rows)

query rows: [('F-RR-120', 1), ('F-MM-320', 1), ('F-CE-110', 1), ('F-CD-202', 1)]


In [3]:
explanation = llm.invoke("Explain this service workload result in two sentences. Do not infer causes or totals outside the rows. "
                         + repr(rows)).content
print("explanation:\n", explanation)

explanation:
 The result lists four service workload entries: F-RR-120, F-MM-320, F-CE-110, and F-CD-202. Each entry has a count of 1.


## b · content-pipeline

We retrieve a local procedure, draft a service bulletin, and run deterministic checks before the draft can leave the pipeline.

In [4]:
from pathlib import Path

procedure_path = Path(data_folder("demo-world/manuals")) / "cd-12-procedure-cd-af-01.md"
procedure = procedure_path.read_text(encoding="utf-8")
draft = llm.invoke(f"Draft a brief internal bulletin for F-CD-101 from this procedure. Cite the source file name {procedure_path.name}, include the part, the two-hour workshop estimate, and the post-repair self test. Do not add instructions. "
                   f"\nSource {procedure_path.name}:\n{procedure}").content
print("draft:\n", draft)

draft:
 **Internal bulletin — F-CD-101**

For error F-CD-101 (engine intake pressure low), the specified part is **CD-AF-01 engine air filter**. Allow **2 hours in the workshop**. After repair, run the self test; the error must not return.

Source: `cd-12-procedure-cd-af-01.md`


In [5]:
checks = {
    "source named": procedure_path.name in draft,
    "part named": "CD-AF-01" in draft,
    "time retained": "2 hour" in draft.lower() or "two hour" in draft.lower(),
    "self test retained": "self test" in draft.lower() or "self-test" in draft.lower(),
}
print("checks:", checks)
print("release:", "ready for editor" if all(checks.values()) else "hold for correction")

checks: {'source named': True, 'part named': True, 'time retained': True, 'self test retained': True}
release: ready for editor


The checks establish that required details appear in the draft. An editor still compares each claim with the retrieved procedure before publication.

## Things to try

- Change the SQL grouping from fault code to mechanic
- Remove one required detail from the prompt and inspect the pipeline result